In [1]:
from google.colab import files
uploaded = files.upload()


Saving retail_sales_dataset.csv to retail_sales_dataset.csv


In [2]:
import pandas as pd

df = pd.read_csv("retail_sales_dataset.csv")
df.head()


,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100


In [3]:
import numpy as np
df["Date"] = pd.to_datetime(df["Date"])
df = df[df["Date"].dt.year <= pd.Timestamp.today().year]
df["YearMonth"] = df["Date"].dt.to_period("M").dt.to_timestamp()

monthly = df.groupby("YearMonth")["Total Amount"].sum().reset_index()
monthly = monthly.sort_values("YearMonth").reset_index(drop=True)


In [4]:
monthly["month"] = monthly["YearMonth"].dt.month
monthly["month_index"] = np.arange(1, len(monthly) + 1)
monthly["prev_month_revenue"] = monthly["Total Amount"].shift(1)
monthly["rolling_avg_3"] = monthly["Total Amount"].rolling(3).mean()
monthly = monthly.dropna().reset_index(drop=True)


In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

X = monthly[["month", "month_index", "prev_month_revenue", "rolling_avg_3"]]
y = monthly["Total Amount"]

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

y_pred = model.predict(X)
r2 = round(r2_score(y, y_pred), 3)
rmse = round(np.sqrt(mean_squared_error(y, y_pred)), 2)

print("✅ Full-data R² Score:", r2)
print("✅ Full-data RMSE:", rmse)


✅ Full-data R² Score: 0.849
✅ Full-data RMSE: 3181.59


In [6]:
last = monthly.iloc[-1]

next_input = pd.DataFrame([{
    "month": (last["month"] % 12) + 1,
    "month_index": last["month_index"] + 1,
    "prev_month_revenue": last["Total Amount"],
    "rolling_avg_3": monthly["Total Amount"].tail(3).mean()
}])

forecast = model.predict(next_input)[0]
forecast_rounded = round(forecast, 2)
lower = round(forecast * 0.9, 2)
upper = round(forecast * 1.1, 2)

print("📈 Predicted Revenue for Next Month: £", forecast_rounded)
print("📊 Confidence Interval: £", lower, "– £", upper)


📈 Predicted Revenue for Next Month: £ 35323.65
📊 Confidence Interval: £ 31791.29 – £ 38856.02


In [7]:
import joblib

joblib.dump(model, "revenue_predictor.pkl")
monthly.tail(3)[["YearMonth", "Total Amount"]].to_csv("last_3_months.csv", index=False)

with open("model_metrics.txt", "w") as f:
    f.write(f"r2={r2}\nrmse={rmse}\n")

with open("forecast_meta.txt", "w") as f:
    f.write(f"predicted={forecast_rounded}\nlower={lower}\nupper={upper}\nr2={r2}\nrmse={rmse}\n")


In [8]:
files.download("revenue_predictor.pkl")
files.download("last_3_months.csv")
files.download("model_metrics.txt")
files.download("forecast_meta.txt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>